# Modbus 클라이언트 실습

서버 노트북(`http://localhost:8889`)에서 Modbus 서버를 먼저 기동한 후 이 노트북을 실행하세요.

```
[이 노트북: client-lab]
  AsyncModbusTcpClient
        │
        ▼  TCP 접속 (server-lab:5020)
  [서버 노트북: server-lab]
  StartAsyncTcpServer (0.0.0.0:5020)
```

### 실습 항목
| 단계 | 항목 |
|------|------|
| Step 1 | 서버 연결 확인 |
| Step 2 | 비트(Coil) 읽기 |
| Step 3 | 비트(Coil) 단일 쓰기 |
| Step 4 | 비트(Coil) 다중 쓰기 |
| Step 5 | 워드(Holding Register) 읽기 |
| Step 6 | 워드(Holding Register) 단일 쓰기 |
| Step 7 | 워드(Holding Register) 다중 쓰기 |
| Step 8 | 종합 실습 |
| Step 9 | 연결 종료 |

---
## [Step 1] 라이브러리 임포트 및 서버 연결

In [1]:
import pymodbus
from pymodbus.client import ModbusTcpClient

print(f"pymodbus 버전: {pymodbus.__version__}")
print("임포트 완료")

pymodbus 버전: 3.10.0
임포트 완료


In [2]:
# 서버 컨테이너 이름 = docker-compose 서비스 이름 'server-lab'
# Docker 내부 DNS가 자동으로 컨테이너 IP로 해석해 줌
HOST = "210.119.14.56"
PORT = 502
DEVICE_ID = 1   # Unit ID (pymodbus 3.10.0: slave → device_id)

client = ModbusTcpClient(HOST, port=PORT)
# await client.connect()
client.connect()

if client.connected:
    print(f"연결 성공! → {HOST}:{PORT}")
else:
    print("연결 실패 — 서버 노트북에서 Step 3 셀을 먼저 실행하세요.")

연결 성공! → 210.119.14.56:502


---
## [Step 2] 비트(Coil) 읽기

**FC 01 — Read Coils**

- 메서드: `read_coils(address, *, count=1, device_id=DEVICE_ID)`
- `count`, `device_id` 는 키워드 전용 인자 (`*` 뒤에 위치)
- 반환: `result.bits` — bool 리스트 (16의 배수로 패딩됨)
- 실제 읽은 개수만큼 슬라이싱: `result.bits[:count]`

In [4]:
COUNT = 10
result = client.read_coils(0, count=COUNT, device_id=DEVICE_ID)

if result.isError():
    print("오류:", result)
else:
    bits = result.bits[:COUNT]
    print("[FC01] 코일 읽기 — 주소 0~9")
    print("-" * 30)
    for i, v in enumerate(bits):
        bar = "■" if v else "□"
        print(f"  코일[{i}] = {bar} {'ON ' if v else 'OFF'}")

[FC01] 코일 읽기 — 주소 0~9
------------------------------
  코일[0] = □ OFF
  코일[1] = □ OFF
  코일[2] = □ OFF
  코일[3] = □ OFF
  코일[4] = □ OFF
  코일[5] = □ OFF
  코일[6] = □ OFF
  코일[7] = □ OFF
  코일[8] = ■ ON 
  코일[9] = □ OFF


---
## [Step 3] 비트(Coil) 단일 쓰기

**FC 05 — Write Single Coil**

- 메서드: `write_coil(address, value, *, device_id=DEVICE_ID)`
- `value`: `True` (ON) 또는 `False` (OFF)

In [12]:
# 켜기
# client.write_coil(1, 1)
# 끄기
client.write_coil(1, 0)

In [14]:
# 0번 코일을 ON으로 변경
result = client.write_coil(0, True, device_id=DEVICE_ID)
if result.isError():
    print("쓰기 오류:", result)
else:
    print("[FC05] 코일[0] = True (ON) 쓰기 완료")

# 1번 코일을 OFF로 변경
result = client.write_coil(1, False, device_id=DEVICE_ID)
if not result.isError():
    print("[FC05] 코일[1] = False (OFF) 쓰기 완료")

# 결과 확인
result = client.read_coils(0, count=5, device_id=DEVICE_ID)
bits = result.bits[:5]
print(f"\n확인) 코일 0~4: {[int(b) for b in bits]}")

[FC05] 코일[0] = True (ON) 쓰기 완료
[FC05] 코일[1] = False (OFF) 쓰기 완료

확인) 코일 0~4: [1, 0, 1, 1, 0]


---
## [Step 4] 비트(Coil) 다중 쓰기

**FC 15 — Write Multiple Coils**

- 메서드: `write_coils(address, values, *, device_id=DEVICE_ID)`
- `values`: bool 리스트

In [8]:
# 코일 0~9번에 교대 패턴 쓰기
values = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1,1,1,1,1,1,1]

import time as tt

client.write_coils(0, values, device_id=DEVICE_ID)
tt.sleep(2)
# 결과 확인
result = client.read_coils(0, count=16, device_id=DEVICE_ID)
bits = result.bits
print(f"확인) 코일 : {[int(b) for b in bits]}")
    
# if result.isError():
#     print("쓰기 오류:", result)
# else:
#     print(f"[FC15] 코일 0~9 다중 쓰기 완료: {[int(v) for v in values]}")



확인) 코일 : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


---
## [Step 5] 워드(Holding Register) 읽기

**FC 03 — Read Holding Registers**

- 메서드: `read_holding_registers(address, *, count=1, device_id=DEVICE_ID)`
- 반환: `result.registers` — int 리스트 (16비트 값, 0~65535)

> 서버 노트북에서 [Step 7] 레지스터 값 변경 셀을 실행한 후 아래 셀을 다시 실행해 보세요.

In [25]:
COUNT = 10
result =  client.read_holding_registers(0, count=COUNT, device_id=DEVICE_ID)

if result.isError():
    print("오류:", result)
else:
    regs = result.registers
    print("[FC03] 홀딩 레지스터 읽기 — 주소 0~9")
    print("-" * 30)
    for i, v in enumerate(regs):
        print(f"  레지스터[{i}] = {v}")

[FC03] 홀딩 레지스터 읽기 — 주소 0~9
------------------------------
  레지스터[0] = 250
  레지스터[1] = 1013
  레지스터[2] = 1500
  레지스터[3] = 400
  레지스터[4] = 500
  레지스터[5] = 1000
  레지스터[6] = 2000
  레지스터[7] = 3000
  레지스터[8] = 4000
  레지스터[9] = 5000


---
## [Step 6] 워드(Holding Register) 단일 쓰기

**FC 06 — Write Single Register**

- 메서드: `write_register(address, value, *, device_id=DEVICE_ID)`
- `value`: 정수 (0~65535)

In [27]:
# 0번 레지스터에 1234 쓰기
result = client.write_register(0, 1234, device_id=DEVICE_ID)
if result.isError():
    print("쓰기 오류:", result)
else:
    print("[FC06] 레지스터[0] = 1234 쓰기 완료")

# 5번 레지스터에 9999 쓰기
result = client.write_register(5, 9999, device_id=DEVICE_ID)
if not result.isError():
    print("[FC06] 레지스터[5] = 9999 쓰기 완료")

# 결과 확인
result = client.read_holding_registers(0, count=10, device_id=DEVICE_ID)
print(f"\n확인) 레지스터 0~9: {result.registers}")

[FC06] 레지스터[0] = 1234 쓰기 완료
[FC06] 레지스터[5] = 9999 쓰기 완료

확인) 레지스터 0~9: [1234, 1013, 1500, 400, 500, 9999, 2000, 3000, 4000, 5000]


---
## [Step 7] 워드(Holding Register) 다중 쓰기

**FC 16 — Write Multiple Registers**

- 메서드: `write_registers(address, values, *, device_id=DEVICE_ID)`
- `values`: 정수 리스트

In [11]:
# 레지스터 0~9번에 10 단위 증가 값 쓰기
values = [0,0,0,0,0,0,0,0,0,0]
result = client.write_registers(0, values, device_id=DEVICE_ID)

if result.isError():
    print("쓰기 오류:", result)
else:
    print(f"[FC16] 레지스터 0~9 다중 쓰기 완료: {values}")

# 결과 확인
result = client.read_holding_registers(0, count=10, device_id=DEVICE_ID)
print(f"확인) 레지스터 0~9: {result.registers}")

[FC16] 레지스터 0~9 다중 쓰기 완료: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
확인) 레지스터 0~9: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
import time as tt
COUNT = 5
counter =[0,0,0,0,0]
for n in range(1000):
    result = client.read_discrete_inputs(5, count=COUNT, device_id=DEVICE_ID)
    bits = result.bits[:COUNT]
    box = [0 if v else 1 for v in bits]
    counter = [c + int(b) for c, b in zip(counter, bits)]
    print(f"\r{box}//{counter}",end="")
    tt.sleep(1)

In [ ]:
COUNT = 5
prev = None
counter = [0] * COUNT

while True:
    result = client.read_discrete_inputs(5, count=COUNT, device_id=DEVICE_ID)
    bits = [0 if v else 1 for v in result.bits[:COUNT]]   # NC 접점

    if prev is not None and bits != prev:      # 변경된 경우만
        counter = [c + (p == 0 and b == 1) for c, p, b in zip(counter, prev, bits)]
        print(f"\r{bits}//{counter}",end="")
        client.write_registers(0, counter, device_id=DEVICE_ID)
    prev = bits
    tt.sleep(0.05)      # 50ms

[1, 1, 1, 1, 1]//[10916, 16538, 10010, 13319, 11982]